# Model Evaluation — Bias / Variance Diagnosis

Answers two questions:
1. **Is the transformer overfitting** (high variance) — big train vs test gap?
2. **Is poor test performance from something else** — bad features, data shift, noise ceiling?

Sections:
1. Prerequisites check
2. Train / Val / Test metrics gap
3. Confidence distribution
4. Calibration (reliability diagram)
5. Learning curves (logistic regression proxy)
6. Complexity ladder — Elo-only → logistic → MLP → transformer
7. Padding analysis (how much history do teams actually have?)
8. Error analysis (what does the model get wrong?)
9. Automated diagnosis


In [ ]:
import sys, pickle, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import torch, yaml
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss, accuracy_score

from src.models.predictor import ValorantPredictor
from src.data.dataset import MatchDataset
from src.training.metrics import compute_all

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

DATA_DIR = Path('../data/processed')
CKPT     = Path('../checkpoints/best_model.pt')
TEMP_F   = Path('../checkpoints/temperature.json')
CFG_F    = Path('../configs/model_config.yaml')

missing = [p for p in [
    DATA_DIR/'train_samples.pkl', DATA_DIR/'val_samples.pkl',
    DATA_DIR/'test_samples.pkl',  CKPT
] if not p.exists()]

if missing:
    print('\u274c Missing files. Run the pipeline first:')
    print('  python scripts/build_dataset.py')
    print('  python scripts/train.py')
    for m in missing: print(f'  missing: {m}')
else:
    print('\u2705 All files found')


## 1. Load Data & Model

In [ ]:
def load_pkl(path):
    with open(path, 'rb') as f: return pickle.load(f)

train_s = load_pkl(DATA_DIR / 'train_samples.pkl')
val_s   = load_pkl(DATA_DIR / 'val_samples.pkl')
test_s  = load_pkl(DATA_DIR / 'test_samples.pkl')
print(f'Train: {len(train_s)}  Val: {len(val_s)}  Test: {len(test_s)}')

with open(CFG_F) as f: model_cfg = yaml.safe_load(f)
device = torch.device('cpu')
model = ValorantPredictor(
    num_scalars     = model_cfg.get('num_scalar_features', 16),
    num_maps        = model_cfg.get('num_maps', 12),
    map_embed_dim   = model_cfg.get('map_embedding_dim', 16),
    d_model         = model_cfg.get('d_model', 64),
    seq_len         = model_cfg.get('seq_len', 20),
    num_heads       = model_cfg.get('num_heads', 4),
    dim_feedforward = model_cfg.get('dim_feedforward', 256),
    num_layers      = model_cfg.get('num_layers', 3),
    num_metas       = model_cfg.get('num_metas', 37),
)
model.load_state_dict(torch.load(CKPT, map_location=device))
model.eval()
print(f'Model loaded  ({model.num_parameters:,} params)')

T = 1.0
if TEMP_F.exists():
    T = json.load(open(TEMP_F)).get('temperature', 1.0)
print(f'Temperature T = {T:.4f}')


In [ ]:
def run_inference(samples, temperature=1.0, batch_size=64):
    ds = MatchDataset(samples)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    logits_all, labels_all = [], []
    with torch.no_grad():
        for batch in loader:
            b = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                 for k, v in batch.items()}
            logits = model(
                b['scalars_a'], b['map_idx_a'], b['pad_mask_a'],
                b['scalars_b'], b['map_idx_b'], b['pad_mask_b'],
                b['meta_idx_a'], b['meta_idx_b'],
                b['elo_a'], b['elo_b'],
            ).squeeze(-1)
            logits_all.append(logits.cpu().numpy())
            labels_all.append(b['label'].cpu().numpy())
    logits = np.concatenate(logits_all)
    probs  = 1 / (1 + np.exp(-logits / temperature))
    labels = np.concatenate(labels_all)
    return probs, labels

print('Running inference on all splits...')
tr_p, tr_l = run_inference(train_s, T)
va_p, va_l = run_inference(val_s,   T)
te_p, te_l = run_inference(test_s,  T)
print('Done')


## 2. Train / Val / Test Metrics Gap

A **large train–test gap** → overfitting (high variance).  
A **small gap but poor test** → noise floor / bad features (high bias).


In [ ]:
splits = [('Train', tr_p, tr_l), ('Val', va_p, va_l), ('Test', te_p, te_l)]
rows = []
for name, p, l in splits:
    m = compute_all(p, l)
    rows.append({'Split': name, 'N': len(l), **{k: round(v, 4) for k, v in m.items()}})
df_gaps = pd.DataFrame(rows).set_index('Split')
print(df_gaps.to_string())

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
metrics_plot = [
    ('accuracy',    'Accuracy',   0.45, 0.75, True),
    ('roc_auc',     'ROC-AUC',    0.45, 0.75, True),
    ('brier_score', 'Brier Score',0.15, 0.28, False),
    ('log_loss',    'Log-loss',   0.58, 0.72, False),
]
colors = ['#3498db', '#2ecc71', '#e74c3c']
for ax, (metric, label, ymin, ymax, higher_better) in zip(axes, metrics_plot):
    vals = [df_gaps.loc[s, metric] for s in ['Train', 'Val', 'Test']]
    bars = ax.bar(['Train','Val','Test'], vals, color=colors, alpha=0.85, edgecolor='#ccc')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + (ymax-ymin)*0.01,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    ax.axhline(vals[0], color='gray', linestyle='--', linewidth=0.8, alpha=0.5,
               label='Train level')
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_ylim(ymin, ymax)
    ax.legend(fontsize=7)
plt.suptitle('Train / Val / Test Gap  (dashed = train level)', fontsize=12)
plt.tight_layout(); plt.show()

acc_gap = df_gaps.loc['Train','accuracy'] - df_gaps.loc['Test','accuracy']
auc_gap = df_gaps.loc['Train','roc_auc']  - df_gaps.loc['Test','roc_auc']
print(f'\nAcc gap (train-test): {acc_gap:+.4f}')
print(f'AUC gap (train-test): {auc_gap:+.4f}')
print(f'  > 0.05 = overfitting signal')


## 3. Confidence Distribution

- Predictions clustered near **0.5** → model is uncertain / not learning signal  
- Predictions **spread or bimodal** → model is confident (correct or not)  


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
split_data = [('Train', tr_p, tr_l, '#3498db'),
              ('Val',   va_p, va_l, '#2ecc71'),
              ('Test',  te_p, te_l, '#e74c3c')]
for ax, (name, probs, labels, color) in zip(axes, split_data):
    ax.hist(probs[labels==1], bins=20, alpha=0.6, label='True win',  color='green',  density=True)
    ax.hist(probs[labels==0], bins=20, alpha=0.6, label='True loss', color='salmon', density=True)
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1)
    confident = np.mean((probs > 0.7) | (probs < 0.3))
    mean_pred = probs.mean()
    ax.text(0.02, 0.94, f'{confident:.0%} confident (>70%)', transform=ax.transAxes, fontsize=8)
    ax.text(0.02, 0.87, f'mean pred: {mean_pred:.3f}',      transform=ax.transAxes, fontsize=8)
    ax.set_xlabel('P(team_a wins)')
    ax.set_ylabel('Density')
    ax.set_title(f'{name}  (n={len(probs)})', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
plt.suptitle('Predicted Probability Distribution', fontsize=12)
plt.tight_layout(); plt.show()


## 4. Calibration (Reliability Diagram)

Diagonal = perfect calibration.  
**Above** diagonal = overconfident. **Below** = underconfident.


In [ ]:
def reliability_diagram(ax, probs, labels, title, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    mids, accs, cnts = [], [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probs >= lo) & (probs < hi)
        if mask.sum() == 0: continue
        mids.append((lo + hi) / 2)
        accs.append(labels[mask].mean())
        cnts.append(mask.sum())
    mids, accs, cnts = np.array(mids), np.array(accs), np.array(cnts)
    ece = np.sum(np.abs(accs - mids) * cnts) / len(probs)
    ax.plot([0,1],[0,1],'k--', linewidth=1, label='Perfect')
    ax.bar(mids, accs, width=0.08, alpha=0.6, color='steelblue')
    for x, y, n in zip(mids, accs, cnts):
        ax.text(x, y + 0.02, str(n), ha='center', fontsize=7, color='#444')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xlabel('Mean predicted prob'); ax.set_ylabel('Actual win rate')
    ax.set_title(f'{title}  (ECE={ece:.3f})', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, p, l, _) in zip(axes, split_data):
    reliability_diagram(ax, p, l, name)
plt.suptitle('Reliability Diagrams', fontsize=12)
plt.tight_layout(); plt.show()


## 5. Learning Curves (Logistic Regression Proxy)

Trains logistic regression on increasing fractions of training data using **mean-pooled scalar features**.  
Much faster than re-training the transformer — same data, different model family.

- Train >> Val gap that **doesn't shrink** as data grows → overfitting  
- Both plateau low → bias / noise ceiling  
- Val keeps improving → more data would help  


In [ ]:
def extract_flat(samples):
    X, y = [], []
    for s in samples:
        def pool(hist):
            if not hist: return np.zeros(16)
            return np.mean([h[:16] for h in hist], axis=0)
        fa = pool(s['history_a'])
        fb = pool(s['history_b'])
        ea, eb = s.get('elo_a', 0.0), s.get('elo_b', 0.0)
        X.append(np.concatenate([fa - fb, fa + fb, [ea - eb, ea + eb]]))
        y.append(int(s['winner'] == 0))
    return np.array(X), np.array(y)

X_tr, y_tr = extract_flat(train_s)
X_va, y_va = extract_flat(val_s)
X_te, y_te = extract_flat(test_s)
sc = StandardScaler().fit(X_tr)
X_tr_s, X_va_s, X_te_s = sc.transform(X_tr), sc.transform(X_va), sc.transform(X_te)
print(f'Feature shape: {X_tr_s.shape}')

fracs = [0.10, 0.20, 0.35, 0.50, 0.70, 1.00]
lc = {'n': [], 'tr_auc': [], 'va_auc': [], 'tr_acc': [], 'va_acc': []}
np.random.seed(42)
for frac in fracs:
    n = max(50, int(len(X_tr_s) * frac))
    idx = np.random.choice(len(X_tr_s), n, replace=False)
    lr = LogisticRegression(max_iter=1000, C=1.0)
    lr.fit(X_tr_s[idx], y_tr[idx])
    lc['n'].append(n)
    lc['tr_auc'].append(roc_auc_score(y_tr[idx], lr.predict_proba(X_tr_s[idx])[:,1]))
    lc['va_auc'].append(roc_auc_score(y_va,      lr.predict_proba(X_va_s)[:,1]))
    lc['tr_acc'].append(accuracy_score(y_tr[idx], lr.predict(X_tr_s[idx])))
    lc['va_acc'].append(accuracy_score(y_va,       lr.predict(X_va_s)))
    print(f'  n={n:4d}  train_auc={lc["tr_auc"][-1]:.3f}  val_auc={lc["va_auc"][-1]:.3f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for ax, tr_k, va_k, ylabel in [(ax1,'tr_auc','va_auc','ROC-AUC'),
                                (ax2,'tr_acc','va_acc','Accuracy')]:
    ax.plot(lc['n'], lc[tr_k], 'o-', color='#3498db', label='Train', linewidth=2)
    ax.plot(lc['n'], lc[va_k], 's-', color='#e74c3c', label='Val',   linewidth=2)
    ax.fill_between(lc['n'], lc[tr_k], lc[va_k], alpha=0.12, color='orange')
    ax.set_xlabel('Training samples'); ax.set_ylabel(ylabel)
    ax.set_title(f'Learning Curve — {ylabel}  (logistic proxy)',
                 fontsize=10, fontweight='bold')
    ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print(f'\nGap at full data:  AUC {lc["tr_auc"][-1]-lc["va_auc"][-1]:+.3f}')
print(f'Val curve slope (last 2 points): {lc["va_auc"][-1]-lc["va_auc"][-2]:+.3f}')
print('  (positive = val still improving with more data)')


## 6. Complexity Ladder

Same flat features, five model families — from least to most complex.  
If a simpler model **matches the transformer** → sequence modeling isn't helping on these features.


In [ ]:
clfs = [
    ('Random',           DummyClassifier(strategy='most_frequent')),
    ('Elo-only logistic', LogisticRegression(max_iter=500)),
    ('Full logistic',     LogisticRegression(max_iter=1000, C=0.5)),
    ('MLP (64-32)',       MLPClassifier(hidden_layer_sizes=(64,32),  max_iter=500,
                                        early_stopping=True, random_state=42)),
    ('MLP (128-64-32)',   MLPClassifier(hidden_layer_sizes=(128,64,32), max_iter=500,
                                        early_stopping=True, random_state=42)),
]

X_tr_elo = X_tr_s[:, -2:]  # just elo_diff and elo_sum
X_va_elo = X_va_s[:, -2:]
X_te_elo = X_te_s[:, -2:]

rows_c = []
for name, clf in clfs:
    elo_only = 'Elo' in name
    Xf, Xv, Xt = (X_tr_elo, X_va_elo, X_te_elo) if elo_only else (X_tr_s, X_va_s, X_te_s)
    clf.fit(Xf, y_tr)
    def sk_m(clf, X, y):
        p = clf.predict_proba(X)[:,1]
        return (round(accuracy_score(y, clf.predict(X)),3),
                round(roc_auc_score(y, p),3),
                round(brier_score_loss(y, p),3))
    tr_m = sk_m(clf, Xf, y_tr)
    va_m = sk_m(clf, Xv, y_va)
    te_m = sk_m(clf, Xt, y_te)
    rows_c.append({'Model': name,
                   'Tr Acc': tr_m[0], 'Va Acc': va_m[0], 'Te Acc': te_m[0],
                   'Tr AUC': tr_m[1], 'Va AUC': va_m[1], 'Te AUC': te_m[1],
                   'Te Brier': te_m[2]})
    print(f'{name:<22}  val_acc={va_m[0]:.3f}  val_auc={va_m[1]:.3f}  test_auc={te_m[1]:.3f}')

# Append transformer
tr_t = compute_all(tr_p, tr_l); va_t = compute_all(va_p, va_l); te_t = compute_all(te_p, te_l)
rows_c.append({'Model': 'Transformer (ours)',
               'Tr Acc': round(tr_t['accuracy'],3), 'Va Acc': round(va_t['accuracy'],3),
               'Te Acc': round(te_t['accuracy'],3),
               'Tr AUC': round(tr_t['roc_auc'],3),  'Va AUC': round(va_t['roc_auc'],3),
               'Te AUC': round(te_t['roc_auc'],3),  'Te Brier': round(te_t['brier_score'],3)})
print(f'Transformer (ours)    val_acc={va_t["accuracy"]:.3f}  '
      f'val_auc={va_t["roc_auc"]:.3f}  test_auc={te_t["roc_auc"]:.3f}')

df_c = pd.DataFrame(rows_c).set_index('Model')
print('\n', df_c[['Va Acc','Te Acc','Va AUC','Te AUC','Te Brier']].to_string())

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(df_c))
ax.bar(x - 0.2, df_c['Va AUC'],  0.35, label='Val AUC',  color='#3498db', alpha=0.85)
ax.bar(x + 0.2, df_c['Te AUC'],  0.35, label='Test AUC', color='#e74c3c', alpha=0.85)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='Random')
for i, (va, te) in enumerate(zip(df_c['Va AUC'], df_c['Te AUC'])):
    ax.text(i-0.2, va+0.005, f'{va:.3f}', ha='center', fontsize=7.5)
    ax.text(i+0.2, te+0.005, f'{te:.3f}', ha='center', fontsize=7.5)
ax.set_xticks(x); ax.set_xticklabels(df_c.index, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('ROC-AUC'); ax.set_ylim(0.45, 0.80)
ax.set_title('Complexity Ladder: Val vs Test AUC', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


## 7. Padding Analysis — How Much History Do Teams Have?

Heavy padding means the transformer processes mostly zeros.  
If most sequences are short, the transformer gains little over a simple mean-pool.


In [ ]:
def hist_lens(samples):
    return ([len(s['history_a']) for s in samples]
          + [len(s['history_b']) for s in samples])

tr_lens = hist_lens(train_s)
te_lens = hist_lens(test_s)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, lens, title in [(axes[0], tr_lens, 'Train'), (axes[1], te_lens, 'Test')]:
    ax.hist(lens, bins=21, range=(-0.5, 20.5), color='#3498db', alpha=0.8, edgecolor='white')
    ax.axvline(np.mean(lens), color='red', linestyle='--', label=f'Mean={np.mean(lens):.1f}')
    full = sum(l == 20 for l in lens)
    few  = sum(l < 5  for l in lens)
    ax.text(0.60, 0.90, f'{full/len(lens):.0%} fully populated (20)',
            transform=ax.transAxes, fontsize=8.5)
    ax.text(0.60, 0.83, f'{few/len(lens):.0%} sparse (<5 matches)',
            transform=ax.transAxes, fontsize=8.5, color='red')
    ax.set_xlabel('History length (out of 20)')
    ax.set_ylabel('Count (team-sides)')
    ax.set_title(f'{title} — History Lengths', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
plt.suptitle('How padded are the sequences? (20 = full history)', fontsize=11)
plt.tight_layout(); plt.show()


## 8. Error Analysis

High-confidence wrong predictions reveal systematic failure modes.


In [ ]:
df_e = pd.DataFrame({
    'team_a':    [s['team_a'] for s in test_s],
    'team_b':    [s['team_b'] for s in test_s],
    'date':      [s['date']   for s in test_s],
    'prob':      te_p,
    'label':     te_l.astype(int),
    'predicted': (te_p > 0.5).astype(int),
})
df_e['correct']    = df_e['predicted'] == df_e['label']
df_e['confidence'] = np.abs(df_e['prob'] - 0.5)
wrong = df_e[~df_e['correct']].sort_values('confidence', ascending=False)
print(f'Wrong: {(~df_e["correct"]).sum()}/{len(df_e)} = {(~df_e["correct"]).mean():.1%}')
print(f'High-confidence errors (conf > 0.3): {(wrong["confidence"]>0.3).sum()}')
print('\nTop 10 most confident wrong predictions:')
print(wrong.head(10)[['date','team_a','team_b','prob','label']].to_string(index=False))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Accuracy by confidence bucket
conf_bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5]
buck_acc, buck_n, buck_lbl = [], [], []
for lo, hi in zip(conf_bins[:-1], conf_bins[1:]):
    mask = (df_e['confidence'] >= lo) & (df_e['confidence'] < hi)
    if mask.sum():
        buck_acc.append(df_e.loc[mask, 'correct'].mean())
        buck_n.append(mask.sum())
    else:
        buck_acc.append(np.nan); buck_n.append(0)
    buck_lbl.append(f'{lo:.1f}-{hi:.1f}')
ax1.bar(range(len(buck_lbl)), buck_acc, color='steelblue', alpha=0.8)
for i, (a, n) in enumerate(zip(buck_acc, buck_n)):
    if not np.isnan(a): ax1.text(i, a + 0.01, f'n={n}', ha='center', fontsize=8)
ax1.axhline(0.5, color='red', linestyle='--', linewidth=1, label='Random')
ax1.set_xticks(range(len(buck_lbl))); ax1.set_xticklabels(buck_lbl)
ax1.set_xlabel('Confidence bucket (|p - 0.5|)')
ax1.set_ylabel('Accuracy'); ax1.set_ylim(0, 1)
ax1.set_title('Test — Accuracy by Confidence', fontsize=10, fontweight='bold')
ax1.legend(fontsize=8)

# Monthly accuracy over time
df_e['ym'] = df_e['date'].str[:7]
monthly = df_e.groupby('ym')['correct'].agg(['mean','count']).reset_index()
ax2.plot(range(len(monthly)), monthly['mean'], 'o-', color='#e74c3c', linewidth=2)
ax2.fill_between(range(len(monthly)), monthly['mean'], 0.5,
                 where=monthly['mean'] >= 0.5, alpha=0.12, color='green')
ax2.fill_between(range(len(monthly)), monthly['mean'], 0.5,
                 where=monthly['mean'] < 0.5, alpha=0.12, color='red')
ax2.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
step = max(1, len(monthly)//8)
ax2.set_xticks(range(0, len(monthly), step))
ax2.set_xticklabels(monthly['ym'].iloc[::step], rotation=35, ha='right', fontsize=7)
ax2.set_ylabel('Monthly Accuracy'); ax2.set_ylim(0.2, 1.0)
ax2.set_title('Test — Accuracy Over Time', fontsize=10, fontweight='bold')
plt.tight_layout(); plt.show()


## 9. Automated Diagnosis

| Signal | Cause | Fix |
|---|---|---|
| Train AUC >> Test AUC (gap >0.08) | Overfitting | More dropout, fewer layers, more data |
| Both plateau ~0.62 AUC | Noise ceiling on current features | Better features (roster, agent picks, econ) |
| Logistic ≈ Transformer AUC | Sequence model not helping | Transformer adds no value; use MLP |
| Val > Test | Distribution shift 2024→2025 | Retrain with `--train-cutoff 2025-06-01` |
| >30% sequences have <5 matches | Sparse history in test | Elo feature becomes critical; cold-start fix |


In [ ]:
print('=' * 55)
print('AUTOMATED DIAGNOSIS')
print('=' * 55)

tr_auc = df_gaps.loc['Train','roc_auc']
va_auc = df_gaps.loc['Val',  'roc_auc']
te_auc = df_gaps.loc['Test', 'roc_auc']
auc_gap = tr_auc - te_auc

print(f'Train AUC: {tr_auc:.3f}  Val AUC: {va_auc:.3f}  Test AUC: {te_auc:.3f}')
print(f'Gap (train-test): {auc_gap:+.3f}')
print()

if auc_gap > 0.08:
    print('[HIGH VARIANCE / OVERFITTING]')
    print('  -> Large train-test gap. Model memorising training set.')
    print('  -> Try: increase dropout, reduce num_layers, add weight decay')
elif te_auc < 0.58:
    print('[HIGH BIAS / NOISE CEILING]')
    print('  -> Model barely beats random. Features are the bottleneck.')
    print('  -> Try: player-level stats, agent comps, economy data')
else:
    print('[MODERATE — near feature ceiling]')
    print('  -> Gap is small; model has learned most available signal.')
    print('  -> Squeeze more: better features > bigger model')

print()
best_simple = df_c.loc[[m for m in df_c.index if 'Transformer' not in m], 'Te AUC'].max()
tfm_auc = df_c.loc['Transformer (ours)', 'Te AUC']
delta = tfm_auc - best_simple
print(f'Best simple model test AUC : {best_simple:.3f}')
print(f'Transformer test AUC       : {tfm_auc:.3f}  (delta {delta:+.3f})')
if delta < 0.01:
    print('  -> Transformer adds NO benefit over simple baselines')
    print('     The sequence structure is not being exploited.')
    print('     Consider: (a) simpler MLP, (b) better sequence features')
elif delta < 0.03:
    print('  -> Transformer adds MARGINAL benefit (< 0.03 AUC)')
else:
    print('  -> Transformer adds MEANINGFUL benefit — keep it')

print()
sparse = sum(l < 5 for l in te_lens) / len(te_lens)
if sparse > 0.25:
    print(f'[PADDING WARNING] {sparse:.0%} of test sequences have <5 matches')
    print('  -> Elo feature becomes the primary signal for these teams')

if va_auc > te_auc + 0.03:
    print()
    print('[DISTRIBUTION SHIFT] Val AUC >> Test AUC')
    print('  -> Model trained on pre-2025 Elo/meta, tested on 2025+')
    print('  -> Retrain: python scripts/build_dataset.py --train-cutoff 2025-06-01')

print('\nDone.')
